# Signal Benchmark Analysis

This notebook analyzes `events.csv` files emitted by the Signal benchmark harness. It focuses on participant-count plateaus, pairwise session setup, prekey material, and message encryption/decryption costs.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
root = Path("benchmark_output")
files = sorted(root.glob("*/events.csv"))
if not files:
    raise FileNotFoundError("No events.csv files found under benchmark_output/*")

df = pd.concat(
    (pd.read_csv(path).assign(source_run_dir=path.parent.name) for path in files),
    ignore_index=True,
)

for col in ["wall_ns", "participant_count", "conversation_size", "prekey_bundle_count", "session_count", "ratchet_step_count", "ciphertext_bytes", "plaintext_bytes", "artifact_size_bytes"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["wall_ms"] = df["wall_ns"] / 1_000_000
df.head()


In [ ]:
bad_markers = ["_com" + "mit", "wel" + "come"]
observed_forbidden = sorted(
    op for op in set(df["op"].dropna().astype(str))
    if any(marker in op.lower() for marker in bad_markers)
)
assert not observed_forbidden, f"Found non-Signal operation names: {observed_forbidden}"

expected_signal_ops = {
    "RegisterParticipant",
    "GeneratePrekeyBundle",
    "PublishPrekeyBundle",
    "EstablishSessions",
    "EncryptMessage",
    "DecryptMessage",
    "RemoveParticipants",
}
latest_dir = sorted(df["source_run_dir"].dropna().unique())[-1]
missing_from_latest = expected_signal_ops.difference(set(df[df["source_run_dir"] == latest_dir]["op"]))
print("Loaded files:", len(files))
print("Rows:", len(df))
print("Latest run dir:", latest_dir)
print("Run IDs sample:", sorted(df["run_id"].dropna().unique())[:10])
print("Missing expected ops in latest run:", sorted(missing_from_latest))


In [ ]:
op_summary = (
    df.groupby(["run_id", "op"], dropna=False)
      .agg(
          rows=("op", "size"),
          wall_ms_median=("wall_ms", "median"),
          wall_ms_p95=("wall_ms", lambda s: s.quantile(0.95)),
          wall_ms_max=("wall_ms", "max"),
      )
      .reset_index()
      .sort_values(["run_id", "op"])
)
op_summary


In [ ]:
plateau = (
    df.dropna(subset=["conversation_size"])
      .groupby(["run_id", "conversation_size", "op"], dropna=False)
      .agg(
          rows=("op", "size"),
          wall_ms_median=("wall_ms", "median"),
          wall_ms_p95=("wall_ms", lambda s: s.quantile(0.95)),
          session_count_median=("session_count", "median"),
          ratchet_step_median=("ratchet_step_count", "median"),
      )
      .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))
for op, part in plateau[plateau["op"].isin(["EncryptMessage", "DecryptMessage", "EstablishSessions"])].groupby("op"):
    part = part.sort_values("conversation_size")
    ax.plot(part["conversation_size"], part["wall_ms_median"], marker="o", label=op)
ax.set_xlabel("Active Signal participants")
ax.set_ylabel("Median wall time (ms)")
ax.set_title("Signal operation cost by conversation size")
ax.legend()
plt.show()
plateau


In [ ]:
messages = df[df["op"].isin(["EncryptMessage", "DecryptMessage"])].copy()
messages["ciphertext_over_plaintext"] = messages["ciphertext_bytes"] / messages["plaintext_bytes"]
message_summary = (
    messages.groupby(["run_id", "conversation_size", "op"], dropna=False)
            .agg(
                rows=("op", "size"),
                plaintext_bytes_median=("plaintext_bytes", "median"),
                ciphertext_bytes_median=("ciphertext_bytes", "median"),
                ciphertext_over_plaintext_median=("ciphertext_over_plaintext", "median"),
                wall_ms_median=("wall_ms", "median"),
            )
            .reset_index()
            .sort_values(["run_id", "conversation_size", "op"])
)
message_summary


In [ ]:
prekeys = df[df["op"].isin(["GeneratePrekeyBundle", "PublishPrekeyBundle"])].copy()
prekey_summary = (
    prekeys.groupby(["run_id", "op"], dropna=False)
           .agg(
               rows=("op", "size"),
               prekey_bundle_count_median=("prekey_bundle_count", "median"),
               artifact_size_bytes_median=("artifact_size_bytes", "median"),
               wall_ms_median=("wall_ms", "median"),
           )
           .reset_index()
           .sort_values(["run_id", "op"])
)
prekey_summary
